In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [36]:
# load the dataset

df=pd.read_csv("Bank Customer Churn Prediction.csv")

In [37]:
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [38]:
df.tail()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
9995,15606229,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,15569892,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,15584532,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,15682355,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
9999,15628319,792,France,Female,28,4,130142.79,1,1,0,38190.78,0


In [39]:
df.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [40]:
# preprocess the data

df = df.drop(columns=['customer_id'])

In [41]:
df = pd.get_dummies(df, columns=['country', 'gender'], drop_first=True)

In [42]:
X = df.drop(columns=['churn'])
y = df['churn']

In [43]:
# Split into training and testing sets (80% train, 20% test)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [44]:
# Scale the numerical features so they have a mean of 0 and variance of 1

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [45]:
# Build the ANN Model

model = Sequential([
    # Input layer + First hidden layer
    # 16 neurons, using ReLU activation function
    Dense(units=16, activation='relu', input_shape=(X_train.shape[1],)),
    
    # Second hidden layer
    # 8 neurons, using ReLU activation function
    Dense(units=8, activation='relu'),
    
    # Output layer
    # 1 neuron with Sigmoid activation because this is a binary classification (0 or 1)
    Dense(units=1, activation='sigmoid')
])

C:\Users\Thor fin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
# Compile & Train the Model

model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

In [47]:
# Train the model

history = model.fit(
    X_train, 
    y_train, 
    validation_split=0.2, 
    batch_size=32, 
    epochs=30, 
    verbose=1
)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7897 - loss: 0.5486 - val_accuracy: 0.7975 - val_loss: 0.4785
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8006 - loss: 0.4607 - val_accuracy: 0.8188 - val_loss: 0.4297
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8159 - loss: 0.4304 - val_accuracy: 0.8238 - val_loss: 0.4145
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8211 - loss: 0.4182 - val_accuracy: 0.8225 - val_loss: 0.4072
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8245 - loss: 0.4075 - val_accuracy: 0.8263 - val_loss: 0.3976
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8359 - loss: 0.3924 - val_accuracy: 0.8400 - val_loss: 0.3837
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8458 - loss: 0.3752 - val_accuracy: 0.8406 - val_loss: 0.3712
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8495 - loss: 0.3631 - val_accuracy: 0.

In [48]:
# Evaluate the Model

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Loss: 0.3405
Test Accuracy: 86.15%


In [49]:
y_pred_prob = model.predict(X_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [50]:
y_pred = (y_pred_prob > 0.5).astype(int)

In [51]:
print(confusion_matrix(y_test, y_pred))

[[1544   63]
 [ 214  179]]


In [52]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1607
           1       0.74      0.46      0.56       393

    accuracy                           0.86      2000
   macro avg       0.81      0.71      0.74      2000
weighted avg       0.85      0.86      0.85      2000



In [53]:
# Save Results

model.save("bank_churn_model.keras")